Total S1 entities in ground truth
Missing/empty match lists
Number of S1 entities with:- 0 matches
- 1 match
- 2+ matches

S2 match count distribution
S3 match count distribution
Total S2/S3 matches
Min/mean/median/p90/p95/p99/max matches per S1
S1 entities with unusually high match counts
Invalid/malformed IDs
Duplicate matched IDs within the same S1 row
Duplicate S1 IDs
S2/S3-only matching statistics
Combined match-count distribution
Separate S2 and S3 reports
A row-level ground_truth_eda.csv
CSV reports useful for the next blocking experiment

In [1]:
from pathlib import Path
from collections import Counter
import re

import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

GROUND_TRUTH_PATH = Path(
    r"E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\dataset\train\train_ground_truth.tsv"
)

OUTPUT_DIR = Path(
    r"E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\student_resource\amazon_er\outputs\eda_outputs\eda_ground_truth_outputs"
)

CHUNK_SIZE = 500_000


# ============================================================
# HELPERS
# ============================================================

S1_PATTERN = re.compile(r"^S1-\d+$")
S2_PATTERN = re.compile(r"^S2-\d+$")
S3_PATTERN = re.compile(r"^S3-\d+$")

VALID_PATTERNS = {
    "S1": S1_PATTERN,
    "S2": S2_PATTERN,
    "S3": S3_PATTERN,
}


def percentile_stats(series):
    """
    Return useful distribution statistics for a numeric Series.
    """
    if len(series) == 0:
        return {
            "count": 0,
            "min": 0,
            "max": 0,
            "mean": 0,
            "median": 0,
            "std": 0,
            "p25": 0,
            "p75": 0,
            "p90": 0,
            "p95": 0,
            "p99": 0,
        }

    values = series.to_numpy()

    return {
        "count": int(len(values)),
        "min": int(np.min(values)),
        "max": int(np.max(values)),
        "mean": float(np.mean(values)),
        "median": float(np.median(values)),
        "std": float(np.std(values)),
        "p25": float(np.percentile(values, 25)),
        "p75": float(np.percentile(values, 75)),
        "p90": float(np.percentile(values, 90)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
    }


def print_section(title):
    print()
    print("=" * 70)
    print(title)
    print("=" * 70)


def safe_split_matches(value):
    """
    Convert matched_entity_ids into a list of IDs.

    Handles:
        NaN
        empty strings
        whitespace
        comma-separated IDs
    """
    if pd.isna(value):
        return []

    value = str(value).strip()

    if not value:
        return []

    return [
        item.strip()
        for item in value.split(",")
        if item.strip()
    ]


def classify_id(entity_id):
    """
    Classify an entity ID as S1/S2/S3/INVALID.
    """
    if pd.isna(entity_id):
        return "INVALID"

    entity_id = str(entity_id).strip()

    for source, pattern in VALID_PATTERNS.items():
        if pattern.fullmatch(entity_id):
            return source

    return "INVALID"


# ============================================================
# MAIN
# ============================================================

def main():

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print_section("LOADING GROUND TRUTH")

    print(f"Input : {GROUND_TRUTH_PATH}")
    print(f"Output: {OUTPUT_DIR}")

    if not GROUND_TRUTH_PATH.exists():
        raise FileNotFoundError(
            f"Ground truth file not found:\n{GROUND_TRUTH_PATH}"
        )

    # --------------------------------------------------------
    # Read in chunks so the script remains memory-friendly.
    # --------------------------------------------------------

    chunks = []

    for chunk in pd.read_csv(
        GROUND_TRUTH_PATH,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        chunksize=CHUNK_SIZE,
    ):
        chunks.append(chunk)

    df = pd.concat(chunks, ignore_index=True)

    print(f"Rows    : {len(df):,}")
    print(f"Columns : {len(df.columns)}")
    print(f"Columns : {list(df.columns)}")

    # --------------------------------------------------------
    # Validate expected columns
    # --------------------------------------------------------

    expected_columns = {
        "source1_entity_id",
        "matched_entity_ids",
    }

    missing_columns = expected_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing expected columns: {missing_columns}"
        )

    # ========================================================
    # 1. BASIC STRUCTURE
    # ========================================================

    print_section("1. BASIC STRUCTURE")

    duplicate_s1_ids = df["source1_entity_id"].duplicated().sum()

    missing_s1_ids = (
        df["source1_entity_id"]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(f"Total ground-truth rows : {len(df):,}")
    print(f"Duplicate S1 IDs        : {duplicate_s1_ids:,}")
    print(f"Missing S1 IDs          : {missing_s1_ids:,}")

    # ========================================================
    # 2. S1 ID VALIDATION
    # ========================================================

    print_section("2. S1 ID VALIDATION")

    s1_ids = df["source1_entity_id"].astype(str).str.strip()

    valid_s1 = s1_ids.str.fullmatch(r"S1-\d+")

    invalid_s1_count = (~valid_s1).sum()

    print(f"Valid S1 IDs   : {valid_s1.sum():,}")
    print(f"Invalid S1 IDs : {invalid_s1_count:,}")

    if invalid_s1_count:
        invalid_s1 = (
            df.loc[~valid_s1, ["source1_entity_id"]]
            .drop_duplicates()
        )

        invalid_s1.to_csv(
            OUTPUT_DIR / "invalid_s1_ids.csv",
            index=False
        )

    # ========================================================
    # 3. PARSE MATCHED IDS
    # ========================================================

    print_section("3. PARSING MATCHED ENTITY IDS")

    df["matched_list"] = df["matched_entity_ids"].apply(
        safe_split_matches
    )

    df["total_matches"] = df["matched_list"].str.len()

    df["s2_matches"] = df["matched_list"].apply(
        lambda ids: sum(
            classify_id(x) == "S2"
            for x in ids
        )
    )

    df["s3_matches"] = df["matched_list"].apply(
        lambda ids: sum(
            classify_id(x) == "S3"
            for x in ids
        )
    )

    df["invalid_matches"] = df["matched_list"].apply(
        lambda ids: sum(
            classify_id(x) == "INVALID"
            for x in ids
        )
    )

    # ========================================================
    # 4. MATCH PRESENCE
    # ========================================================

    print_section("4. MATCH PRESENCE")

    zero_matches = (df["total_matches"] == 0).sum()
    one_match = (df["total_matches"] == 1).sum()
    multiple_matches = (df["total_matches"] > 1).sum()

    total_s1 = len(df)

    print(f"S1 entities                  : {total_s1:,}")
    print(
        f"S1 entities with 0 matches   : "
        f"{zero_matches:,} "
        f"({zero_matches / total_s1 * 100:.2f}%)"
    )
    print(
        f"S1 entities with 1 match     : "
        f"{one_match:,} "
        f"({one_match / total_s1 * 100:.2f}%)"
    )
    print(
        f"S1 entities with 2+ matches  : "
        f"{multiple_matches:,} "
        f"({multiple_matches / total_s1 * 100:.2f}%)"
    )

    # ========================================================
    # 5. MATCH SOURCE DISTRIBUTION
    # ========================================================

    print_section("5. MATCH SOURCE DISTRIBUTION")

    total_s2_matches = int(df["s2_matches"].sum())
    total_s3_matches = int(df["s3_matches"].sum())
    total_matches = int(df["total_matches"].sum())

    print(f"Total matches : {total_matches:,}")
    print(f"S2 matches    : {total_s2_matches:,}")
    print(f"S3 matches    : {total_s3_matches:,}")

    if total_matches:
        print(
            f"S2 share      : "
            f"{total_s2_matches / total_matches * 100:.2f}%"
        )

        print(
            f"S3 share      : "
            f"{total_s3_matches / total_matches * 100:.2f}%"
        )

    # ========================================================
    # 6. MATCH COUNT DISTRIBUTION
    # ========================================================

    print_section("6. TOTAL MATCH COUNT DISTRIBUTION")

    distribution = (
        df["total_matches"]
        .value_counts()
        .sort_index()
        .rename_axis("match_count")
        .reset_index(name="s1_entity_count")
    )

    distribution["percentage"] = (
        distribution["s1_entity_count"]
        / total_s1
        * 100
    )

    print(distribution.to_string(index=False))

    distribution.to_csv(
        OUTPUT_DIR / "total_match_count_distribution.csv",
        index=False
    )

    # ========================================================
    # 7. S2 MATCH COUNT DISTRIBUTION
    # ========================================================

    print_section("7. S2 MATCH COUNT DISTRIBUTION")

    s2_distribution = (
        df["s2_matches"]
        .value_counts()
        .sort_index()
        .rename_axis("s2_match_count")
        .reset_index(name="s1_entity_count")
    )

    s2_distribution["percentage"] = (
        s2_distribution["s1_entity_count"]
        / total_s1
        * 100
    )

    print(s2_distribution.to_string(index=False))

    s2_distribution.to_csv(
        OUTPUT_DIR / "s2_match_count_distribution.csv",
        index=False
    )

    # ========================================================
    # 8. S3 MATCH COUNT DISTRIBUTION
    # ========================================================

    print_section("8. S3 MATCH COUNT DISTRIBUTION")

    s3_distribution = (
        df["s3_matches"]
        .value_counts()
        .sort_index()
        .rename_axis("s3_match_count")
        .reset_index(name="s1_entity_count")
    )

    s3_distribution["percentage"] = (
        s3_distribution["s1_entity_count"]
        / total_s1
        * 100
    )

    print(s3_distribution.to_string(index=False))

    s3_distribution.to_csv(
        OUTPUT_DIR / "s3_match_count_distribution.csv",
        index=False
    )

    # ========================================================
    # 9. MATCH COUNT STATISTICS
    # ========================================================

    print_section("9. MATCH COUNT STATISTICS")

    total_stats = percentile_stats(df["total_matches"])
    s2_stats = percentile_stats(df["s2_matches"])
    s3_stats = percentile_stats(df["s3_matches"])

    stats_df = pd.DataFrame(
        [
            {"source": "TOTAL", **total_stats},
            {"source": "S2", **s2_stats},
            {"source": "S3", **s3_stats},
        ]
    )

    print(stats_df.to_string(index=False))

    stats_df.to_csv(
        OUTPUT_DIR / "match_count_statistics.csv",
        index=False
    )

    # ========================================================
    # 10. S2/S3 PRESENCE COMBINATIONS
    # ========================================================

    print_section("10. S2/S3 PRESENCE COMBINATIONS")

    def presence_category(row):
        has_s2 = row["s2_matches"] > 0
        has_s3 = row["s3_matches"] > 0

        if has_s2 and has_s3:
            return "S2 + S3"
        elif has_s2:
            return "S2 only"
        elif has_s3:
            return "S3 only"
        else:
            return "No matches"

    df["match_source_category"] = df.apply(
        presence_category,
        axis=1
    )

    source_presence = (
        df["match_source_category"]
        .value_counts()
        .rename_axis("category")
        .reset_index(name="s1_entity_count")
    )

    source_presence["percentage"] = (
        source_presence["s1_entity_count"]
        / total_s1
        * 100
    )

    print(source_presence.to_string(index=False))

    source_presence.to_csv(
        OUTPUT_DIR / "match_source_presence.csv",
        index=False
    )

    # ========================================================
    # 11. HIGH-MULTIPLICITY ENTITIES
    # ========================================================

    print_section("11. HIGH-MULTIPLICITY ENTITIES")

    top_multi = (
        df[
            [
                "source1_entity_id",
                "s2_matches",
                "s3_matches",
                "total_matches",
                "matched_entity_ids",
            ]
        ]
        .sort_values(
            "total_matches",
            ascending=False
        )
        .head(100)
    )

    print(top_multi.head(20).to_string(index=False))

    top_multi.to_csv(
        OUTPUT_DIR / "top_100_high_multiplicity_entities.csv",
        index=False
    )

    # ========================================================
    # 12. INVALID MATCH IDS
    # ========================================================

    print_section("12. INVALID MATCH IDS")

    invalid_match_rows = df["invalid_matches"] > 0

    print(
        f"Rows containing invalid match IDs : "
        f"{invalid_match_rows.sum():,}"
    )

    print(
        f"Total invalid match IDs           : "
        f"{df['invalid_matches'].sum():,}"
    )

    if invalid_match_rows.any():

        invalid_records = []

        for _, row in df.loc[
            invalid_match_rows,
            [
                "source1_entity_id",
                "matched_entity_ids",
            ],
        ].iterrows():

            ids = safe_split_matches(
                row["matched_entity_ids"]
            )

            invalid_ids = [
                x for x in ids
                if classify_id(x) == "INVALID"
            ]

            invalid_records.append(
                {
                    "source1_entity_id":
                        row["source1_entity_id"],
                    "invalid_ids":
                        ",".join(invalid_ids),
                }
            )

        pd.DataFrame(invalid_records).to_csv(
            OUTPUT_DIR / "invalid_match_ids.csv",
            index=False
        )

    # ========================================================
    # 13. DUPLICATE MATCH IDS WITHIN A ROW
    # ========================================================

    print_section("13. DUPLICATE MATCH IDS WITHIN ROW")

    def duplicate_count(ids):
        return len(ids) - len(set(ids))

    df["duplicate_match_ids"] = df["matched_list"].apply(
        duplicate_count
    )

    duplicate_rows = (
        df["duplicate_match_ids"] > 0
    )

    print(
        f"Rows containing duplicate match IDs : "
        f"{duplicate_rows.sum():,}"
    )

    print(
        f"Total duplicate occurrences         : "
        f"{df['duplicate_match_ids'].sum():,}"
    )

    if duplicate_rows.any():

        df.loc[
            duplicate_rows,
            [
                "source1_entity_id",
                "matched_entity_ids",
                "duplicate_match_ids",
            ],
        ].to_csv(
            OUTPUT_DIR / "duplicate_match_ids.csv",
            index=False
        )

    # ========================================================
    # 14. GLOBAL MATCHED ENTITY ID DUPLICATION
    # ========================================================

    print_section("14. GLOBAL MATCHED ENTITY ID DUPLICATION")

    matched_counter = Counter()

    for ids in df["matched_list"]:
        matched_counter.update(ids)

    matched_frequency = pd.DataFrame(
        matched_counter.items(),
        columns=[
            "matched_entity_id",
            "s1_entity_count",
        ],
    )

    duplicated_across_s1 = (
        matched_frequency[
            matched_frequency["s1_entity_count"] > 1
        ]
        .sort_values(
            "s1_entity_count",
            ascending=False
        )
    )

    print(
        f"Unique matched entity IDs : "
        f"{len(matched_frequency):,}"
    )

    print(
        f"Matched IDs appearing for >1 S1 : "
        f"{len(duplicated_across_s1):,}"
    )

    print(
        duplicated_across_s1.head(20).to_string(
            index=False
        )
    )

    matched_frequency.to_csv(
        OUTPUT_DIR / "matched_entity_frequency.csv",
        index=False
    )

    duplicated_across_s1.to_csv(
        OUTPUT_DIR / "matched_ids_across_multiple_s1.csv",
        index=False
    )

    # ========================================================
    # 15. MATCHED ID SOURCE VALIDATION
    # ========================================================

    print_section("15. MATCHED ID SOURCE VALIDATION")

    source_counts = Counter()

    for ids in df["matched_list"]:
        for entity_id in ids:
            source_counts[classify_id(entity_id)] += 1

    source_validation = pd.DataFrame(
        [
            {
                "source": source,
                "count": count,
            }
            for source, count
            in sorted(source_counts.items())
        ]
    )

    print(source_validation.to_string(index=False))

    source_validation.to_csv(
        OUTPUT_DIR / "matched_id_source_validation.csv",
        index=False
    )

    # ========================================================
    # 16. S1-LEVEL ANALYSIS TABLE
    # ========================================================

    print_section("16. CREATING S1-LEVEL ANALYSIS TABLE")

    s1_analysis = df[
        [
            "source1_entity_id",
            "s2_matches",
            "s3_matches",
            "total_matches",
            "invalid_matches",
            "duplicate_match_ids",
            "match_source_category",
        ]
    ].copy()

    s1_analysis["has_s2_match"] = (
        s1_analysis["s2_matches"] > 0
    )

    s1_analysis["has_s3_match"] = (
        s1_analysis["s3_matches"] > 0
    )

    s1_analysis["has_any_match"] = (
        s1_analysis["total_matches"] > 0
    )

    s1_analysis.to_csv(
        OUTPUT_DIR / "s1_ground_truth_analysis.csv",
        index=False
    )

    # ========================================================
    # 17. SUMMARY CSV
    # ========================================================

    print_section("17. SAVING SUMMARY")

    summary = {
        "total_s1_rows": total_s1,
        "duplicate_s1_ids": int(duplicate_s1_ids),
        "invalid_s1_ids": int(invalid_s1_count),

        "s1_with_zero_matches": int(zero_matches),
        "s1_with_one_match": int(one_match),
        "s1_with_multiple_matches": int(multiple_matches),

        "total_matches": total_matches,
        "total_s2_matches": total_s2_matches,
        "total_s3_matches": total_s3_matches,

        "s1_with_s2_match": int(
            (df["s2_matches"] > 0).sum()
        ),
        "s1_with_s3_match": int(
            (df["s3_matches"] > 0).sum()
        ),
        "s1_with_both_s2_s3": int(
            (
                (df["s2_matches"] > 0)
                &
                (df["s3_matches"] > 0)
            ).sum()
        ),

        "rows_with_invalid_matches": int(
            invalid_match_rows.sum()
        ),
        "total_invalid_match_ids": int(
            df["invalid_matches"].sum()
        ),

        "rows_with_duplicate_match_ids": int(
            duplicate_rows.sum()
        ),
        "total_duplicate_match_occurrences": int(
            df["duplicate_match_ids"].sum()
        ),

        "unique_matched_entity_ids": int(
            len(matched_frequency)
        ),

        "matched_ids_used_by_multiple_s1": int(
            len(duplicated_across_s1)
        ),
    }

    summary_df = pd.DataFrame(
        [
            {
                "metric": key,
                "value": value,
            }
            for key, value in summary.items()
        ]
    )

    summary_df.to_csv(
        OUTPUT_DIR / "ground_truth_summary.csv",
        index=False
    )

    # ========================================================
    # 18. FINAL CONSOLE SUMMARY
    # ========================================================

    print_section("GROUND TRUTH EDA COMPLETE")

    print(f"S1 entities                  : {total_s1:,}")
    print(f"S1 entities with 0 matches   : {zero_matches:,}")
    print(f"S1 entities with 1 match     : {one_match:,}")
    print(f"S1 entities with 2+ matches  : {multiple_matches:,}")

    print()
    print(f"Total S2 matches             : {total_s2_matches:,}")
    print(f"Total S3 matches             : {total_s3_matches:,}")
    print(f"Total matches                : {total_matches:,}")

    print()
    print(
        f"S1 with S2 match             : "
        f"{(df['s2_matches'] > 0).sum():,}"
    )

    print(
        f"S1 with S3 match             : "
        f"{(df['s3_matches'] > 0).sum():,}"
    )

    print(
        f"S1 with both S2 + S3        : "
        f"{(
            (df['s2_matches'] > 0)
            &
            (df['s3_matches'] > 0)
        ).sum():,}"
    )

    print()
    print(f"Output directory: {OUTPUT_DIR}")

    print()
    print("Generated files:")

    for file in sorted(OUTPUT_DIR.glob("*.csv")):
        print(f"  {file.name}")


if __name__ == "__main__":
    main()


LOADING GROUND TRUTH
Input : E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\dataset\train\train_ground_truth.tsv
Output: E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\student_resource\amazon_er\outputs\eda_outputs\eda_ground_truth_outputs
Rows    : 2,206,821
Columns : 2
Columns : ['source1_entity_id', 'matched_entity_ids']

1. BASIC STRUCTURE
Total ground-truth rows : 2,206,821
Duplicate S1 IDs        : 0
Missing S1 IDs          : 0

2. S1 ID VALIDATION
Valid S1 IDs   : 2,206,821
Invalid S1 IDs : 0

3. PARSING MATCHED ENTITY IDS

4. MATCH PRESENCE
S1 entities                  : 2,206,821
S1 entities with 0 matches   : 123,247 (5.58%)
S1 entities with 1 match     : 119,157 (5.40%)
S1 entities with 2+ matches  : 1,964,417 (89.02%)

5. MATCH SOURCE DISTRIBUTION
Total matches : 7,638,365
S2 matches    : 3,693,619
S3 matches    : 3,944,746
S2 share      : 48.36%
S3 share      : 51.64%

6. TOTAL MATCH COUNT DISTRIBUTION
 match_count  s1_entity_count  percentage
           0  